# Checkpoint 2: PR Merge Outcome Modeling

This notebook turns the Checkpoint 1 audit into a reproducible modeling baseline. It keeps the scope deliberately conservative: only PR-level features judged plausibly available before the final merge decision are used, while identifiers, clear post-outcome fields, and unresolved timing-sensitive variables remain excluded.


## Objective and scope

**Research question:** Can PR-level features help explain and predict PR merge outcomes on GitHub?

Checkpoint 2 focuses on the first serious supervised baseline. The provided test split is still reserved for the final evaluation; this notebook uses an internal validation split from the training file.


In [ ]:
from __future__ import annotations

import hashlib
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import Markdown, display
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.frozen import FrozenEstimator
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupShuffleSplit, RepeatedStratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180

RANDOM_STATE = 42
TARGET_COLUMN = "merged_or_not"
TARGET_LABELS = {0: "Not merged", 1: "Merged"}


def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "data" / "raw").exists():
            return candidate
    fallback = Path("/Users/mahmoudali/Documents/ADES - first project")
    if fallback.exists():
        return fallback
    raise FileNotFoundError("Could not locate the ADES project root.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
CHECKPOINT2_DIR = PROJECT_ROOT / "deliverables" / "checkpoint-2"
FINAL_DIR = PROJECT_ROOT / "deliverables" / "final"
FIGURE_DIR = FINAL_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "prfeatures_train_data.csv"
TEST_PATH = DATA_DIR / "prfeatures_test_data.csv"

metadata_columns = ["project_id", "creator_id", "last_close_time"]

exclude_ids = ["id", "project_id", "creator_id", "last_closer_id"]
exclude_post_outcome = ["last_close_time", "lifetime_minutes", "reopen_or_not"]
ambiguous_features = [
    "num_comments", "has_comments", "num_participants", "core_comment",
    "contrib_comment", "inte_comment", "has_exchange", "at_tag",
    "num_code_comments", "num_code_comments_con", "perc_neg_emotion",
    "perc_pos_emotion", "comment_conflict", "contrib_open", "contrib_cons",
    "contrib_extra", "contrib_agree", "contrib_neur", "inte_open",
    "inte_cons", "inte_extra", "inte_agree", "inte_neur",
    "perc_contrib_pos_emo", "perc_contrib_neg_emo", "perc_inte_pos_emo",
    "perc_inte_neg_emo", "social_strength", "same_user", "ci_build_num",
    "ci_failed_perc", "integrator_availability",
]
headline_safe_features = [
    "first_pr", "core_member", "prior_interaction",
    "followers", "prev_pullreqs", "account_creation_days",
    "contrib_perc_commit", "sloc", "team_size", "language",
    "open_issue_num", "project_age", "open_pr_num", "fork_num",
    "test_lines_per_kloc", "stars",
    "test_cases_per_kloc", "asserts_per_kloc", "perc_external_contribs",
    "churn_addition", "churn_deletion", "description_length",
    "test_inclusion", "ci_exists", "friday_effect",
]

t0_creation_features = [
    "first_pr", "core_member", "prior_interaction",
    "followers", "prev_pullreqs", "account_creation_days",
    "contrib_perc_commit", "sloc", "team_size", "language",
    "open_issue_num", "project_age", "open_pr_num", "fork_num",
    "test_lines_per_kloc", "stars",
    "test_cases_per_kloc", "asserts_per_kloc", "perc_external_contribs",
    "description_length", "ci_exists", "friday_effect",
]

ultra_conservative_features = [
    "first_pr", "core_member", "prior_interaction",
    "followers", "prev_pullreqs", "account_creation_days",
    "language", "project_age", "description_length",
    "test_inclusion", "ci_exists", "friday_effect",
]

integrator_assumed_extension_features = [
    "prior_review_num",
]

timing_assumed_extension_features = [
    "pr_succ_rate", "requester_succ_rate", "num_commits",
    "src_churn", "files_changed", "test_churn",
]

review_process_features = [
    "num_comments", "has_comments", "num_participants", "core_comment",
    "contrib_comment", "inte_comment", "has_exchange", "at_tag",
    "num_code_comments", "num_code_comments_con", "perc_neg_emotion",
    "perc_pos_emotion", "comment_conflict", "contrib_open", "contrib_cons",
    "contrib_extra", "contrib_agree", "contrib_neur", "inte_open",
    "inte_cons", "inte_extra", "inte_agree", "inte_neur",
    "perc_contrib_pos_emo", "perc_contrib_neg_emo", "perc_inte_pos_emo",
    "perc_inte_neg_emo", "social_strength", "same_user", "ci_build_num",
    "ci_failed_perc", "integrator_availability", "prior_review_num",
]

t2_review_process_features = [
    *headline_safe_features,
    *review_process_features,
]

prediction_contracts = {
    "T0_creation": {
        "features": t0_creation_features,
        "label": "PR creation",
        "interpretation": "Earliest defensible model: contributor history, project snapshots, description length, CI presence, and calendar context only.",
    },
    "T1_diff": {
        "features": headline_safe_features,
        "label": "Submitted diff",
        "interpretation": "Headline early-information model: T0 plus initial PR diff/test-inclusion signals under the stated timing contract.",
    },
    "T2_review_process": {
        "features": t2_review_process_features,
        "label": "Review process",
        "interpretation": "Late-stage process model: adds comments, discussion, CI-progress, and reviewer/integrator context after review has started.",
    },
}

integrator_assumed_features = [
    *headline_safe_features,
    *integrator_assumed_extension_features,
]

extended_timing_assumed_features = [
    *headline_safe_features,
    *timing_assumed_extension_features,
]

# Backward-compatible aliases used by the Checkpoint 2 notebook and helper
# defaults. In the final notebook, these represent the headline leakage-safer policy.
candidate_safe_features = headline_safe_features
questionable_timing_features = timing_assumed_extension_features
strict_safe_features = headline_safe_features

safe_usecols = [TARGET_COLUMN, *headline_safe_features]
ultra_conservative_usecols = [TARGET_COLUMN, *ultra_conservative_features]
integrator_assumed_usecols = [TARGET_COLUMN, *integrator_assumed_features]
extended_usecols = [TARGET_COLUMN, *extended_timing_assumed_features]
analysis_usecols = [
    TARGET_COLUMN,
    *dict.fromkeys([
        *extended_timing_assumed_features,
        *integrator_assumed_extension_features,
        *review_process_features,
    ]),
]
analysis_load_usecols = [*metadata_columns, *analysis_usecols]
strict_safe_usecols = safe_usecols

binary_features = [
    "first_pr", "core_member", "test_inclusion", "ci_exists", "friday_effect"
]
categorical_features = ["language"]
numeric_features = [
    feature for feature in candidate_safe_features
    if feature not in binary_features and feature not in categorical_features
]

safe_feature_timing_reason = {
    "first_pr": "Contributor history flag available before or at PR submission.",
    "prior_review_num": "Integrator/reviewer history count; held out of the headline model because it may require knowing the eventual reviewer/integrator before the final decision.",
    "core_member": "Author role/affiliation metadata available before review outcome.",
    "prior_interaction": "Historical author-integrator interaction count before the current PR outcome.",
    "followers": "Contributor profile context available at PR submission.",
    "prev_pullreqs": "Contributor historical PR count before the current PR outcome.",
    "account_creation_days": "Contributor account age at PR submission.",
    "contrib_perc_commit": "Contributor historical share of project commits before the current PR outcome.",
    "sloc": "Project size snapshot available independently of the current PR closure.",
    "team_size": "Project/team context available before the current PR outcome.",
    "language": "Project language category available before modeling; treated as nominal.",
    "open_issue_num": "Project issue-count snapshot used as pre-outcome project context.",
    "project_age": "Repository age at PR submission.",
    "open_pr_num": "Project open-PR workload snapshot; used only under the assumption it is measured at or before PR submission.",
    "fork_num": "Repository popularity/context snapshot available before the current PR outcome.",
    "pr_succ_rate": "Target-adjacent historical project PR success rate; held for timing-assumed sensitivity unless pre-PR computation is verified.",
    "test_lines_per_kloc": "Project test-density snapshot available before closure.",
    "stars": "Repository popularity snapshot available before the current PR outcome.",
    "test_cases_per_kloc": "Project test-density snapshot available before closure.",
    "asserts_per_kloc": "Project test/assertion-density snapshot available before closure.",
    "perc_external_contribs": "Historical project contributor-mix rate before the current PR outcome.",
    "requester_succ_rate": "Target-adjacent requester merge-success rate; held for timing-assumed sensitivity unless pre-PR computation is verified.",
    "churn_addition": "Initial PR diff size available after submission and before final closure.",
    "churn_deletion": "Initial PR diff size available after submission and before final closure.",
    "description_length": "PR description length available at submission.",
    "test_inclusion": "Initial PR diff/test-inclusion flag available before final closure.",
    "ci_exists": "Repository CI configuration/availability flag, not a CI outcome.",
    "test_churn": "PR evolution/test-churn field treated as timing-sensitive because source docs distinguish open-time and close-time variants.",
    "num_commits": "PR evolution field treated as timing-sensitive because source docs describe the unsuffixed version as close-time commit count.",
    "src_churn": "PR evolution field treated as timing-sensitive because source docs distinguish open-time and close-time source churn.",
    "files_changed": "PR evolution field treated as timing-sensitive because source docs distinguish open-time and close-time file-change counts.",
    "friday_effect": "Submission calendar flag available immediately at PR creation.",
}

feature_source_notes = {
    "zenodo_prfeatures": "Zenodo PRFeatures record: 80/20 split, 72 PR-related variables, language label encoding, preprocessing notes.",
    "msr2020_dataset": "Zhang, Rastogi, and Yu MSR 2020 dataset paper: Table 1 describes contributor, project, pull-request, and CI factors.",
    "gitlink_field_readme": "new_pullreq field README: field-level timing notes distinguish submission-time snapshots from close-time PR evolution fields.",
    "tse2022_decisions": "Zhang et al. TSE 2022 PR-decision paper: discusses pull-request decision factors and integrator experience.",
}

feature_timing_evidence_records = {
    "first_pr": ("contributor_history", "pre-pr contributor history", "Contributor first-PR flag before or at PR submission.", "low", "headline", "MSR 2020 Table 1; GitLink README"),
    "core_member": ("contributor_history", "pre-pr contributor/project role", "Contributor core-member flag before review outcome.", "low", "headline", "MSR 2020 Table 1; GitLink README"),
    "prior_interaction": ("contributor_history", "historical interaction", "Contributor-project interaction count from prior three months.", "low", "headline", "MSR 2020 Table 1; GitLink README"),
    "followers": ("contributor_history", "profile snapshot", "Contributor profile context at PR submission.", "low", "headline", "GitLink README"),
    "prev_pullreqs": ("contributor_history", "pre-pr contributor history", "Contributor previous PR count before the current PR.", "low", "headline", "MSR 2020 Table 1; GitLink README"),
    "account_creation_days": ("contributor_history", "pre-pr contributor history", "Account age at PR creation.", "low", "headline", "MSR 2020 Table 1; GitLink README"),
    "contrib_perc_commit": ("contributor_history", "historical contribution share", "Contributor share of project commits before the current PR outcome.", "medium", "headline", "GitLink README"),
    "sloc": ("project_context", "submission-time project snapshot", "Project source lines of code at PR submission.", "low", "headline", "GitLink README"),
    "team_size": ("project_context", "submission-time project snapshot", "Core developer count at PR submission.", "low", "headline", "GitLink README"),
    "language": ("project_context", "project metadata", "Main project language; encoded as nominal category.", "low", "headline", "MSR 2020 Table 1; Zenodo record"),
    "open_issue_num": ("project_context", "submission-time project snapshot", "Open issue count when submitting the PR.", "low", "headline", "MSR 2020 Table 1; GitLink README"),
    "project_age": ("project_context", "submission-time project snapshot", "Project age at PR creation.", "low", "headline", "MSR 2020 Table 1; GitLink README"),
    "open_pr_num": ("project_context", "submission-time project snapshot", "Open PR workload when submitting the PR.", "low", "headline", "MSR 2020 Table 1; GitLink README"),
    "fork_num": ("project_context", "submission-time project snapshot", "Fork count when submitting the PR.", "low", "headline", "MSR 2020 Table 1; GitLink README"),
    "test_lines_per_kloc": ("testing_context", "submission-time project snapshot", "Project test-code density at PR submission.", "low", "headline", "MSR 2020 Table 1; GitLink README"),
    "stars": ("project_context", "submission-time project snapshot", "Repository star/watch count at PR submission.", "low", "headline", "MSR 2020 Table 1; GitLink README"),
    "test_cases_per_kloc": ("testing_context", "submission-time project snapshot", "Project test-case density at PR submission.", "low", "headline", "MSR 2020 Table 1; GitLink README"),
    "asserts_per_kloc": ("testing_context", "submission-time project snapshot", "Project assertion density at PR submission.", "low", "headline", "MSR 2020 Table 1; GitLink README"),
    "perc_external_contribs": ("project_context", "submission-time project snapshot", "External contributor percentage at PR submission.", "low", "headline", "GitLink README"),
    "churn_addition": ("pr_scope", "submission-time PR diff", "Added lines at PR submission according to field README.", "low", "headline", "MSR 2020 Table 1; GitLink README"),
    "churn_deletion": ("pr_scope", "submission-time PR diff", "Deleted lines at PR submission according to field README.", "low", "headline", "MSR 2020 Table 1; GitLink README"),
    "description_length": ("pr_scope", "submission-time PR text", "PR description length available at submission.", "low", "headline", "MSR 2020 Table 1; GitLink README"),
    "test_inclusion": ("testing_context", "submission-time PR diff", "Whether the submitted PR includes test code.", "low", "headline", "MSR 2020 Table 1; GitLink README"),
    "ci_exists": ("testing_context", "CI presence, not CI result", "Whether the PR/repository uses CI; CI outcome fields remain held back.", "medium", "headline", "MSR 2020 Table 1; GitLink README"),
    "friday_effect": ("calendar", "submission-time calendar", "Whether PR was submitted on Friday.", "low", "headline", "GitLink README"),
    "prior_review_num": ("integrator_assumed", "integrator history", "Prior reviews of an integrator; held back because the eventual integrator may be unknown at prediction time.", "high", "sensitivity_only", "MSR 2020 Table 1; TSE 2022"),
    "pr_succ_rate": ("target_adjacent", "historical project outcome rate", "Project PR acceptance rate; held back as target-adjacent unless pre-PR computation is verified.", "high", "sensitivity_only", "MSR 2020 Table 1; GitLink README"),
    "requester_succ_rate": ("target_adjacent", "historical requester outcome rate", "Contributor PR success rate; held back as target-adjacent unless pre-PR computation is verified.", "high", "sensitivity_only", "GitLink README"),
    "num_commits": ("close_time_pr_evolution", "close-time PR state", "Commit count at PR close; not part of headline model.", "high", "sensitivity_only", "GitLink README"),
    "src_churn": ("close_time_pr_evolution", "close-time PR state", "Source churn at PR close; not part of headline model.", "high", "sensitivity_only", "GitLink README"),
    "files_changed": ("close_time_pr_evolution", "close-time PR state", "Changed file count at PR close; not part of headline model.", "high", "sensitivity_only", "GitLink README"),
    "test_churn": ("close_time_pr_evolution", "close-time PR state", "Test churn at PR close; not part of headline model.", "high", "sensitivity_only", "GitLink README"),
}

feature_family_groups = {
    "contributor_history": [
        "first_pr", "core_member", "prior_interaction", "followers",
        "prev_pullreqs", "account_creation_days", "contrib_perc_commit",
    ],
    "project_context": [
        "sloc", "team_size", "open_issue_num", "project_age",
        "open_pr_num", "fork_num", "stars", "perc_external_contribs",
    ],
    "pr_scope": ["churn_addition", "churn_deletion", "description_length"],
    "testing_ci_context": [
        "test_lines_per_kloc", "test_cases_per_kloc", "asserts_per_kloc",
        "test_inclusion", "ci_exists",
    ],
    "language_calendar": ["language", "friday_effect"],
}


def availability_reason(feature_name: str) -> str:
    return safe_feature_timing_reason.get(
        feature_name,
        "Conservative pre-outcome PR-level feature retained for modeling.",
    )


def target_distribution(df: pd.DataFrame, split: str) -> pd.DataFrame:
    counts = df[TARGET_COLUMN].value_counts().sort_index()
    return pd.DataFrame(
        {
            "split": split,
            "target_value": counts.index,
            "label": [TARGET_LABELS[int(value)] for value in counts.index],
            "count": counts.values,
            "percentage": (counts.values / len(df) * 100).round(2),
        }
    )


def stratified_sample(df: pd.DataFrame, n: int | None, random_state: int = RANDOM_STATE) -> pd.DataFrame:
    if n is None or n >= len(df):
        return df.copy()
    _, sample = train_test_split(
        df,
        test_size=n,
        stratify=df[TARGET_COLUMN],
        random_state=random_state,
    )
    return sample.reset_index(drop=True)


def feature_type_groups(feature_list: list[str] | None = None) -> tuple[list[str], list[str], list[str]]:
    selected_features = list(candidate_safe_features if feature_list is None else feature_list)
    selected_binary = [feature for feature in binary_features if feature in selected_features]
    selected_categorical = [feature for feature in categorical_features if feature in selected_features]
    selected_numeric = [
        feature for feature in selected_features
        if feature not in selected_binary and feature not in selected_categorical
    ]
    return selected_numeric, selected_binary, selected_categorical


def make_preprocessor(feature_list: list[str] | None = None) -> ColumnTransformer:
    selected_numeric, selected_binary, selected_categorical = feature_type_groups(feature_list)
    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), selected_numeric),
            ("binary", SimpleImputer(strategy="most_frequent"), selected_binary),
            ("categorical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), selected_categorical),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def build_models(y_train: pd.Series, feature_list: list[str] | None = None) -> dict[str, Pipeline]:
    models: dict[str, Pipeline] = {
        "Dummy majority": Pipeline([
            ("preprocess", make_preprocessor(feature_list)),
            ("model", DummyClassifier(strategy="most_frequent")),
        ]),
        "Logistic regression balanced": Pipeline([
            ("preprocess", make_preprocessor(feature_list)),
            ("model", LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                solver="lbfgs",
                random_state=RANDOM_STATE,
            )),
        ]),
        "Decision tree balanced": Pipeline([
            ("preprocess", make_preprocessor(feature_list)),
            ("model", DecisionTreeClassifier(
                max_depth=10,
                min_samples_leaf=80,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )),
        ]),
        "Random forest balanced": Pipeline([
            ("preprocess", make_preprocessor(feature_list)),
            ("model", RandomForestClassifier(
                n_estimators=80,
                max_depth=14,
                min_samples_leaf=60,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ]),
        "Hist gradient boosting weighted": Pipeline([
            ("preprocess", make_preprocessor(feature_list)),
            ("model", HistGradientBoostingClassifier(
                max_iter=140,
                max_leaf_nodes=31,
                learning_rate=0.08,
                l2_regularization=0.01,
                random_state=RANDOM_STATE,
            )),
        ]),
    }
    return models


def score_merged_probability(pipeline: Pipeline, X: pd.DataFrame) -> np.ndarray:
    if hasattr(pipeline, "predict_proba"):
        return pipeline.predict_proba(X)[:, 1]
    elif hasattr(pipeline[-1], "decision_function"):
        decision = pipeline.decision_function(X)
        return 1 / (1 + np.exp(-decision))
    return pipeline.predict(X).astype(float)


def predict_with_not_merged_threshold(
    pipeline: Pipeline,
    X: pd.DataFrame,
    threshold: float,
) -> np.ndarray:
    score_not_merged = 1 - score_merged_probability(pipeline, X)
    return np.where(score_not_merged >= threshold, 0, 1)


def score_predictions(
    name: str,
    y: pd.Series,
    y_pred: np.ndarray,
    y_score_merged: np.ndarray,
    threshold_label: str = "default_model_threshold",
    threshold_value: float | None = None,
) -> dict[str, object]:
    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    try:
        roc_auc = roc_auc_score(y, y_score_merged)
    except ValueError:
        roc_auc = np.nan
    try:
        average_precision_not_merged = average_precision_score(1 - y, 1 - y_score_merged)
    except ValueError:
        average_precision_not_merged = np.nan

    return {
        "model": name,
        "threshold_label": threshold_label,
        "threshold_value": threshold_value,
        "accuracy": accuracy_score(y, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "precision_not_merged": precision_score(y, y_pred, pos_label=0, zero_division=0),
        "recall_not_merged": recall_score(y, y_pred, pos_label=0, zero_division=0),
        "f1_not_merged": f1_score(y, y_pred, pos_label=0, zero_division=0),
        "precision_merged": precision_score(y, y_pred, pos_label=1, zero_division=0),
        "recall_merged": recall_score(y, y_pred, pos_label=1, zero_division=0),
        "f1_merged": f1_score(y, y_pred, pos_label=1, zero_division=0),
        "roc_auc_merged": roc_auc,
        "average_precision_not_merged": average_precision_not_merged,
        "actual_not_merged_predicted_not_merged": int(cm[0, 0]),
        "actual_not_merged_predicted_merged": int(cm[0, 1]),
        "actual_merged_predicted_not_merged": int(cm[1, 0]),
        "actual_merged_predicted_merged": int(cm[1, 1]),
        "actual_not_merged": int(cm[0, :].sum()),
        "actual_merged": int(cm[1, :].sum()),
        "predicted_not_merged": int(cm[:, 0].sum()),
        "predicted_merged": int(cm[:, 1].sum()),
        "correct_not_merged": int(cm[0, 0]),
        "missed_not_merged": int(cm[0, 1]),
        "false_not_merged": int(cm[1, 0]),
        "correct_merged": int(cm[1, 1]),
    }


def score_pipeline(
    name: str,
    pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    threshold: float | None = None,
    threshold_label: str = "default_model_threshold",
) -> dict[str, object]:
    y_score_merged = score_merged_probability(pipeline, X)
    if threshold is None:
        y_pred = pipeline.predict(X)
        threshold_value = None
    else:
        y_pred = predict_with_not_merged_threshold(pipeline, X, threshold)
        threshold_value = float(threshold)
    return score_predictions(
        name,
        y,
        y_pred,
        y_score_merged,
        threshold_label=threshold_label,
        threshold_value=threshold_value,
    )


def add_metric_metadata(
    row_or_df: dict[str, object] | pd.DataFrame,
    *,
    feature_policy: str,
    feature_count: int,
    source_rows: int,
    training_rows: int,
    validation_rows: int | None,
    training_scope: str,
) -> dict[str, object] | pd.DataFrame:
    metadata = {
        "feature_policy": feature_policy,
        "feature_count": feature_count,
        "source_rows": source_rows,
        "training_rows": training_rows,
        "validation_rows": validation_rows,
        "training_scope": training_scope,
    }
    if isinstance(row_or_df, pd.DataFrame):
        df = row_or_df.copy()
        for column, value in reversed(metadata.items()):
            df.insert(0, column, value)
        return df
    return {**metadata, **row_or_df}


def fit_pipeline(name: str, pipeline: Pipeline, X_train: pd.DataFrame, y_train: pd.Series) -> Pipeline:
    fitted_pipeline = clone(pipeline)
    if name == "Hist gradient boosting weighted":
        sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
        fitted_pipeline.fit(X_train, y_train, model__sample_weight=sample_weight)
    else:
        fitted_pipeline.fit(X_train, y_train)
    return fitted_pipeline


def fit_and_compare(
    train_df: pd.DataFrame,
    model_sample_size: int | None,
    validation_size: float = 0.25,
    feature_list: list[str] | None = None,
    feature_policy: str = "headline_leakage_safer_features",
) -> tuple[pd.DataFrame, dict[str, Pipeline], tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]]:
    selected_features = list(candidate_safe_features if feature_list is None else feature_list)
    model_df = stratified_sample(train_df[[TARGET_COLUMN, *selected_features]], model_sample_size)
    X = model_df[selected_features]
    y = model_df[TARGET_COLUMN]
    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=validation_size,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    models = build_models(y_train, selected_features)
    fitted: dict[str, Pipeline] = {}
    rows: list[dict[str, object]] = []

    for name, pipeline in models.items():
        fitted_pipeline = fit_pipeline(name, pipeline, X_train, y_train)
        fitted[name] = fitted_pipeline
        rows.append(score_pipeline(name, fitted_pipeline, X_valid, y_valid))

    training_scope = (
        "full_source_internal_validation"
        if model_sample_size is None or len(model_df) == len(train_df)
        else f"stratified_sample_{len(model_df)}"
    )
    comparison = pd.DataFrame(rows).sort_values(
        ["f1_not_merged", "balanced_accuracy", "average_precision_not_merged"],
        ascending=False,
    )
    comparison = add_metric_metadata(
        comparison.reset_index(drop=True),
        feature_policy=feature_policy,
        feature_count=len(selected_features),
        source_rows=len(train_df),
        training_rows=len(X_train),
        validation_rows=len(X_valid),
        training_scope=training_scope,
    )
    return comparison, fitted, (X_train, X_valid, y_train, y_valid)


def threshold_tuning_table(pipeline: Pipeline, X_valid: pd.DataFrame, y_valid: pd.Series) -> pd.DataFrame:
    score_not_merged = 1 - score_merged_probability(pipeline, X_valid)
    precision, recall, thresholds = precision_recall_curve(1 - y_valid, score_not_merged)
    table = pd.DataFrame(
        {
            "threshold": thresholds,
            "precision_not_merged": precision[:-1],
            "recall_not_merged": recall[:-1],
        }
    )
    table["f1_not_merged"] = (
        2 * table["precision_not_merged"] * table["recall_not_merged"]
        / (table["precision_not_merged"] + table["recall_not_merged"]).replace(0, np.nan)
    ).fillna(0)
    table = table.sort_values(
        ["f1_not_merged", "recall_not_merged", "precision_not_merged"],
        ascending=[False, False, False],
    )
    return table.reset_index(drop=True)


def repeated_validation_summary(
    train_df: pd.DataFrame,
    model_sample_size: int,
    feature_list: list[str] | None = None,
    feature_policy: str = "headline_leakage_safer_features",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    selected_features = list(candidate_safe_features if feature_list is None else feature_list)
    cv_df = stratified_sample(train_df[[TARGET_COLUMN, *selected_features]], model_sample_size)
    X = cv_df[selected_features]
    y = cv_df[TARGET_COLUMN]
    cv_n_splits = 3
    splitter = RepeatedStratifiedKFold(n_splits=cv_n_splits, n_repeats=2, random_state=RANDOM_STATE)
    rows: list[dict[str, object]] = []
    for fold_number, (train_idx, valid_idx) in enumerate(splitter.split(X, y), start=1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        for name, pipeline in build_models(y_train, selected_features).items():
            fitted_pipeline = fit_pipeline(name, pipeline, X_train, y_train)
            row = score_pipeline(name, fitted_pipeline, X_valid, y_valid)
            row["fold"] = fold_number
            row["feature_policy"] = feature_policy
            row["feature_count"] = len(selected_features)
            row["source_rows"] = len(train_df)
            row["training_rows"] = len(X_train)
            row["validation_rows"] = len(X_valid)
            row["training_scope"] = f"{len(cv_df)}_row_repeated_cv_diagnostic"
            rows.append(row)
    fold_scores = pd.DataFrame(rows)
    metric_columns = [
        "accuracy", "balanced_accuracy", "precision_not_merged",
        "recall_not_merged", "f1_not_merged", "roc_auc_merged",
        "average_precision_not_merged",
    ]
    summary = (
        fold_scores.groupby("model")[metric_columns]
        .agg(["mean", "std"])
        .reset_index()
    )
    summary.columns = [
        column[0] if column[1] == "" else f"{column[0]}_{column[1]}"
        for column in summary.columns.to_flat_index()
    ]
    summary = summary.sort_values(
        [
            "f1_not_merged_mean",
            "balanced_accuracy_mean",
            "average_precision_not_merged_mean",
        ],
        ascending=False,
    ).reset_index(drop=True)
    summary.insert(1, "cv_folds", 6)
    summary.insert(2, "model_sample_size", len(cv_df))
    summary.insert(3, "feature_policy", feature_policy)
    summary.insert(4, "feature_count", len(selected_features))
    summary.insert(5, "source_rows", len(train_df))
    summary.insert(6, "training_rows", len(cv_df) * (cv_n_splits - 1) // cv_n_splits)
    summary.insert(7, "validation_rows", len(cv_df) // cv_n_splits)
    summary.insert(8, "training_scope", f"{len(cv_df)}_row_repeated_cv_diagnostic")
    return summary, fold_scores


def bootstrap_metric_intervals(
    y_true: pd.Series,
    y_pred: np.ndarray,
    y_score_merged: np.ndarray,
    threshold_label: str,
    point_estimates: dict[str, float] | None = None,
    n_resamples: int = 500,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    y_true_array = np.asarray(y_true)
    y_pred_array = np.asarray(y_pred)
    y_score_array = np.asarray(y_score_merged)
    metric_rows: list[dict[str, float | str]] = []
    samples: dict[str, list[float]] = {
        "precision_not_merged": [],
        "recall_not_merged": [],
        "f1_not_merged": [],
        "balanced_accuracy": [],
        "roc_auc_merged": [],
    }
    for _ in range(n_resamples):
        indices = rng.integers(0, len(y_true_array), len(y_true_array))
        y_boot = y_true_array[indices]
        pred_boot = y_pred_array[indices]
        score_boot = y_score_array[indices]
        samples["precision_not_merged"].append(precision_score(y_boot, pred_boot, pos_label=0, zero_division=0))
        samples["recall_not_merged"].append(recall_score(y_boot, pred_boot, pos_label=0, zero_division=0))
        samples["f1_not_merged"].append(f1_score(y_boot, pred_boot, pos_label=0, zero_division=0))
        samples["balanced_accuracy"].append(balanced_accuracy_score(y_boot, pred_boot))
        if len(np.unique(y_boot)) == 2:
            samples["roc_auc_merged"].append(roc_auc_score(y_boot, score_boot))
    for metric, values in samples.items():
        clean_values = np.asarray([value for value in values if not pd.isna(value)])
        metric_rows.append(
            {
                "threshold_label": threshold_label,
                "metric": metric,
                "estimate": float(point_estimates.get(metric, np.mean(clean_values)) if point_estimates else np.mean(clean_values)),
                "bootstrap_mean": float(np.mean(clean_values)),
                "ci_lower_95": float(np.quantile(clean_values, 0.025)),
                "ci_upper_95": float(np.quantile(clean_values, 0.975)),
                "bootstrap_resamples": n_resamples,
            }
        )
    return pd.DataFrame(metric_rows)


def feature_names_from_pipeline(pipeline: Pipeline) -> np.ndarray:
    preprocessor = pipeline.named_steps["preprocess"]
    return preprocessor.get_feature_names_out()


def feature_importance_table(pipeline: Pipeline) -> pd.DataFrame:
    model = pipeline.named_steps["model"]
    names = feature_names_from_pipeline(pipeline)
    if hasattr(model, "feature_importances_"):
        values = model.feature_importances_
    elif hasattr(model, "coef_"):
        values = np.abs(model.coef_[0])
    else:
        return pd.DataFrame(columns=["feature", "importance"])
    return (
        pd.DataFrame({"feature": names, "importance": values})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )


def file_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def raw_data_fingerprint(paths: dict[str, Path]) -> pd.DataFrame:
    rows = []
    for label, path in paths.items():
        header = pd.read_csv(path, nrows=0)
        first_column = header.columns[0]
        row_count = sum(
            len(chunk)
            for chunk in pd.read_csv(
                path,
                usecols=[first_column],
                chunksize=250_000,
                low_memory=False,
            )
        )
        rows.append(
            {
                "dataset": label,
                "path": str(path.relative_to(PROJECT_ROOT)),
                "bytes": path.stat().st_size,
                "rows": row_count,
                "columns": len(header.columns),
                "sha256": file_sha256(path),
                "header_preview": ", ".join(header.columns[:8]),
            }
        )
    return pd.DataFrame(rows)


def feature_timing_evidence_table() -> pd.DataFrame:
    rows = []
    ordered_features = [
        *headline_safe_features,
        *integrator_assumed_extension_features,
        *timing_assumed_extension_features,
    ]
    for feature in ordered_features:
        family, timing, evidence, risk, model_role, source = feature_timing_evidence_records[feature]
        rows.append(
            {
                "feature": feature,
                "feature_family": family,
                "documented_timing": timing,
                "evidence": evidence,
                "risk_level": risk,
                "model_role": model_role,
                "source": source,
            }
        )
    return pd.DataFrame(rows)


def prediction_contract_feature_map_table() -> pd.DataFrame:
    availability_by_contract = {
        "T0_creation": "available at PR creation or from prior contributor/project history",
        "T1_diff": "available after the submitted PR diff is visible, before final closure",
        "T2_review_process": "available only after review/discussion/CI process has started",
    }
    rows = []
    seen: set[tuple[str, str]] = set()
    for contract_name, contract in prediction_contracts.items():
        for feature in contract["features"]:
            key = (contract_name, feature)
            if key in seen:
                continue
            seen.add(key)
            family, timing, evidence, risk, role, source = feature_timing_evidence_records.get(
                feature,
                (
                    "review_process",
                    "during-review process",
                    "Review, discussion, CI-progress, or integrator-context field; excluded from earlier contracts.",
                    "high" if contract_name == "T2_review_process" else "medium",
                    "review_stage",
                    "PRFeatures schema",
                ),
            )
            rows.append(
                {
                    "contract": contract_name,
                    "contract_label": contract["label"],
                    "feature": feature,
                    "feature_family": family,
                    "availability": availability_by_contract[contract_name],
                    "documented_timing": timing,
                    "risk_level": risk,
                    "model_role": role if contract_name != "T2_review_process" else "review_process",
                    "rationale": evidence,
                    "source": source,
                }
            )
    return pd.DataFrame(rows)


def review_process_feature_audit_table() -> pd.DataFrame:
    included = set(review_process_features)
    rows = []
    for feature in [*ambiguous_features, *integrator_assumed_extension_features]:
        rows.append(
            {
                "feature": feature,
                "stage": "T2_review_process" if feature in included else "held_back",
                "included_in_t2": feature in included,
                "excluded_from_early_reason": (
                    "This field can encode review discussion, participant behavior, CI progress, "
                    "or reviewer/integrator context that is unavailable at PR creation."
                ),
                "late_stage_rationale": (
                    "Used only to estimate the extra signal available after review starts."
                    if feature in included
                    else "Still held back because the field is not needed for the review-stage contrast."
                ),
            }
        )
    return pd.DataFrame(rows)


def comment_dataset_profile_table(path: Path) -> pd.DataFrame:
    usecols = [
        "owner_name", "repo_name", "pull_no", "merged_or_not",
        "word_count", "has_code_element", "pos_vr", "neu_vr",
        "neg_vr", "compound", "emotion_vr",
    ]
    comments = pd.read_csv(path, usecols=usecols, low_memory=False)
    repo_keys = comments["owner_name"].astype(str) + "/" + comments["repo_name"].astype(str)
    pr_keys = repo_keys + "#" + comments["pull_no"].astype(str)
    label_counts = comments["merged_or_not"].value_counts(dropna=False).to_dict()
    rows = [
        {
            "metric": "raw_rows",
            "value": len(comments),
            "interpretation": "Comment-level rows available in the separate comment dataset.",
        },
        {
            "metric": "unique_repositories",
            "value": repo_keys.nunique(),
            "interpretation": "Unique owner/repository keys in the comment dataset.",
        },
        {
            "metric": "unique_repo_pr_keys",
            "value": pr_keys.nunique(),
            "interpretation": "Unique owner/repository/pull-number keys in the comment dataset.",
        },
        {
            "metric": "merged_comment_rows",
            "value": int(label_counts.get(1, 0)),
            "interpretation": "Comment rows labeled as merged by the comment dataset.",
        },
        {
            "metric": "not_merged_comment_rows",
            "value": int(label_counts.get(0, 0)),
            "interpretation": "Comment rows labeled as not merged by the comment dataset.",
        },
        {
            "metric": "mean_word_count",
            "value": float(comments["word_count"].mean()),
            "interpretation": "Average comment length in the comment dataset.",
        },
        {
            "metric": "mean_negative_valence",
            "value": float(comments["neg_vr"].mean()),
            "interpretation": "Average negative sentiment score in published comments.",
        },
        {
            "metric": "joined_to_prfeatures",
            "value": "false",
            "interpretation": "The PRFeatures files expose numeric project/PR ids, while the comment file exposes owner/repo/pull number; no reliable local join key is present.",
        },
    ]
    return pd.DataFrame(rows)


def split_overlap_summary(train_df: pd.DataFrame, test_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for column in ["project_id", "creator_id"]:
        train_values = set(train_df[column].unique())
        test_values = set(test_df[column].unique())
        overlap_values = train_values & test_values
        test_overlap_rows = int(test_df[column].isin(overlap_values).sum())
        rows.append(
            {
                "entity": column,
                "train_unique": len(train_values),
                "test_unique": len(test_values),
                "overlap_unique": len(overlap_values),
                "test_unique_overlap_pct": len(overlap_values) / len(test_values) * 100 if test_values else np.nan,
                "test_rows_with_seen_entity": test_overlap_rows,
                "test_rows_with_seen_entity_pct": test_overlap_rows / len(test_df) * 100,
            }
        )
    return pd.DataFrame(rows)


def data_quality_summary(train_df: pd.DataFrame, test_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for split, df in {"train": train_df, "test": test_df}.items():
        rows.extend(
            [
                {"split": split, "check": "rows", "value": len(df)},
                {"split": split, "check": "columns_loaded", "value": df.shape[1]},
                {"split": split, "check": "explicit_null_cells", "value": int(df.isna().sum().sum())},
                {"split": split, "check": "duplicate_loaded_rows", "value": int(df.duplicated().sum())},
                {"split": split, "check": "not_merged_rate_pct", "value": (1 - df[TARGET_COLUMN].mean()) * 100},
                {"split": split, "check": "project_count", "value": df["project_id"].nunique()},
                {"split": split, "check": "creator_count", "value": df["creator_id"].nunique()},
            ]
        )
    return pd.DataFrame(rows)


def numeric_distribution_summary(df: pd.DataFrame, feature_list: list[str]) -> pd.DataFrame:
    numeric = [feature for feature in feature_list if feature != "language"]
    desc = df[numeric].describe(percentiles=[0.5, 0.9, 0.99]).T
    summary = desc[["mean", "std", "50%", "90%", "99%", "max"]].reset_index(names="feature")
    summary["skew"] = df[numeric].skew(numeric_only=True).values
    summary["outlier_flag"] = np.where(summary["99%"] < summary["max"], "extreme_tail", "no_extreme_tail")
    return summary


def class_conditioned_feature_summary(df: pd.DataFrame, feature_list: list[str]) -> pd.DataFrame:
    numeric = [feature for feature in feature_list if feature != "language"]
    rows = []
    grouped = df.groupby(TARGET_COLUMN)
    for feature in numeric:
        for target_value, values in grouped[feature]:
            rows.append(
                {
                    "feature": feature,
                    "target_value": target_value,
                    "target_label": TARGET_LABELS[int(target_value)],
                    "mean": values.mean(),
                    "median": values.median(),
                    "p90": values.quantile(0.9),
                }
            )
    return pd.DataFrame(rows)


def language_project_context_summary(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    language_summary = (
        df.groupby("language")[TARGET_COLUMN]
        .agg(pr_count="size", merge_rate="mean")
        .reset_index()
    )
    language_summary["not_merged_rate"] = 1 - language_summary["merge_rate"]
    language_summary[["merge_rate", "not_merged_rate"]] *= 100

    project_summary = (
        df.groupby("project_id")[TARGET_COLUMN]
        .agg(pr_count="size", merge_rate="mean")
        .reset_index()
    )
    project_summary["not_merged_rate"] = (1 - project_summary["merge_rate"]) * 100
    project_summary["merge_rate"] *= 100
    size_codes = pd.qcut(
        project_summary["pr_count"],
        q=4,
        labels=False,
        duplicates="drop",
    )
    size_labels = ["small", "medium", "large", "very_large"]
    project_summary["size_bin"] = size_codes.map(lambda value: size_labels[int(value)] if pd.notna(value) else "single_bin")
    project_bins = (
        project_summary.groupby("size_bin", observed=False)
        .agg(project_count=("project_id", "size"), median_pr_count=("pr_count", "median"), median_not_merged_rate=("not_merged_rate", "median"))
        .reset_index()
    )
    return language_summary.round(3), project_bins.round(3)


def project_creator_concentration(df: pd.DataFrame, split: str) -> pd.DataFrame:
    rows = []
    for entity in ["project_id", "creator_id"]:
        counts = df[entity].value_counts()
        for top_n in [10, 100, 500]:
            capped_n = min(top_n, len(counts))
            rows.append(
                {
                    "split": split,
                    "entity": entity,
                    "top_n": top_n,
                    "unique_entities": len(counts),
                    "rows_covered": int(counts.head(capped_n).sum()),
                    "row_pct": counts.head(capped_n).sum() / len(df) * 100,
                }
            )
    return pd.DataFrame(rows).round({"row_pct": 3})


def eda_key_findings(
    train_df: pd.DataFrame,
    target_dist: pd.DataFrame,
    concentration: pd.DataFrame,
) -> pd.DataFrame:
    train_targets = target_dist[target_dist["split"] == "train"].set_index("label")

    def not_merged_rate_for(feature: str, value: int) -> float:
        subset = train_df[train_df[feature] == value]
        return (1 - subset[TARGET_COLUMN].mean()) * 100

    def class_median(feature: str, target_value: int) -> float:
        return float(train_df.loc[train_df[TARGET_COLUMN] == target_value, feature].median())

    language_rates = (
        train_df.groupby("language")[TARGET_COLUMN]
        .agg(pr_count="size", merge_rate="mean")
        .assign(not_merged_rate=lambda df: (1 - df["merge_rate"]) * 100)
        .sort_values("not_merged_rate", ascending=False)
    )
    project_top_500 = concentration[
        (concentration["split"] == "train")
        & (concentration["entity"] == "project_id")
        & (concentration["top_n"] == 500)
    ].iloc[0]
    creator_top_500 = concentration[
        (concentration["split"] == "train")
        & (concentration["entity"] == "creator_id")
        & (concentration["top_n"] == 500)
    ].iloc[0]
    most_difficult_language = language_rates.iloc[0]
    least_difficult_language = language_rates.iloc[-1]

    rows = [
        {
            "finding": "Class imbalance drives metric choice",
            "evidence": (
                f"Merged PRs: {int(train_targets.loc['Merged', 'count']):,} "
                f"({train_targets.loc['Merged', 'percentage']:.2f}%); not merged: "
                f"{int(train_targets.loc['Not merged', 'count']):,} "
                f"({train_targets.loc['Not merged', 'percentage']:.2f}%)."
            ),
            "interpretation": "Accuracy-only evaluation would reward the majority class and hide not-merged failures.",
        },
        {
            "finding": "First-time contributors have visibly higher non-merge rates",
            "evidence": (
                f"First PR not-merged rate {not_merged_rate_for('first_pr', 1):.2f}% "
                f"vs {not_merged_rate_for('first_pr', 0):.2f}% for non-first PRs."
            ),
            "interpretation": "Contributor history is a meaningful association, not a causal claim.",
        },
        {
            "finding": "Core-member status separates outcomes",
            "evidence": (
                f"Non-core not-merged rate {not_merged_rate_for('core_member', 0):.2f}% "
                f"vs {not_merged_rate_for('core_member', 1):.2f}% for core members."
            ),
            "interpretation": "Role/context features help explain why contributor-history features matter.",
        },
        {
            "finding": "Merged PRs come from contributors with deeper prior project history",
            "evidence": (
                f"Median previous PRs: {class_median('prev_pullreqs', 0):.0f} for not-merged "
                f"vs {class_median('prev_pullreqs', 1):.0f} for merged; median contribution share "
                f"{class_median('contrib_perc_commit', 0):.3f} vs {class_median('contrib_perc_commit', 1):.3f}."
            ),
            "interpretation": "The model's reliance on contributor-history signals is consistent with EDA.",
        },
        {
            "finding": "Non-merged PRs are associated with larger or busier project contexts",
            "evidence": (
                f"Median open PRs: {class_median('open_pr_num', 0):.0f} for not-merged "
                f"vs {class_median('open_pr_num', 1):.0f} for merged; median stars "
                f"{class_median('stars', 0):.0f} vs {class_median('stars', 1):.0f}."
            ),
            "interpretation": "Project context is predictive, but it also raises external-validity and clustering concerns.",
        },
        {
            "finding": "CI presence is associated with a lower non-merge rate",
            "evidence": (
                f"CI-present not-merged rate {not_merged_rate_for('ci_exists', 1):.2f}% "
                f"vs {not_merged_rate_for('ci_exists', 0):.2f}% when no CI is recorded."
            ),
            "interpretation": "CI is treated as context only; CI outcome fields remain excluded from the headline model.",
        },
        {
            "finding": "Rows are concentrated in recurring projects and creators",
            "evidence": (
                f"Top 500 projects contain {project_top_500['row_pct']:.2f}% of train rows; "
                f"top 500 creators contain {creator_top_500['row_pct']:.2f}%."
            ),
            "interpretation": "This supports project/creator overlap checks and stricter group-holdout stress tests.",
        },
        {
            "finding": "Language-code groups differ, but labels are encoded",
            "evidence": (
                f"Highest not-merged language code {int(most_difficult_language.name)}: "
                f"{most_difficult_language['not_merged_rate']:.2f}%; lowest code "
                f"{int(least_difficult_language.name)}: {least_difficult_language['not_merged_rate']:.2f}%."
            ),
            "interpretation": "Language is useful as nominal project metadata, not as an ordinal numeric scale.",
        },
    ]
    return pd.DataFrame(rows)


def compact_threshold_outputs(threshold_table: pd.DataFrame, max_curve_points: int = 400) -> tuple[pd.DataFrame, pd.DataFrame]:
    top = threshold_table.head(50).copy()
    curve = threshold_table.sort_values("threshold").copy()
    if len(curve) > max_curve_points:
        positions = np.linspace(0, len(curve) - 1, max_curve_points).round().astype(int)
        curve = curve.iloc[np.unique(positions)]
    return top.reset_index(drop=True), curve.reset_index(drop=True)


def threshold_stability_summary(
    train_df: pd.DataFrame,
    model_name: str,
    feature_list: list[str],
    sample_size: int = 120_000,
) -> pd.DataFrame:
    selected_features = list(feature_list)
    cv_df = stratified_sample(train_df[[TARGET_COLUMN, *selected_features]], sample_size, random_state=RANDOM_STATE + 303)
    X = cv_df[selected_features]
    y = cv_df[TARGET_COLUMN]
    splitter = RepeatedStratifiedKFold(n_splits=3, n_repeats=2, random_state=RANDOM_STATE)
    rows = []
    for fold_number, (train_idx, valid_idx) in enumerate(splitter.split(X, y), start=1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        model = fit_pipeline(
            model_name,
            build_models(y_train, selected_features)[model_name],
            X_train,
            y_train,
        )
        default_row = score_pipeline(model_name, model, X_valid, y_valid)
        threshold_table = threshold_tuning_table(model, X_valid, y_valid)
        best_threshold = float(threshold_table.iloc[0]["threshold"])
        tuned_row = score_pipeline(
            model_name,
            model,
            X_valid,
            y_valid,
            threshold=best_threshold,
            threshold_label="fold_tuned_not_merged_f1",
        )
        rows.append(
            {
                "fold": fold_number,
                "model": model_name,
                "model_sample_size": len(cv_df),
                "training_rows": len(X_train),
                "validation_rows": len(X_valid),
                "best_threshold": best_threshold,
                "default_precision_not_merged": default_row["precision_not_merged"],
                "default_recall_not_merged": default_row["recall_not_merged"],
                "default_f1_not_merged": default_row["f1_not_merged"],
                "tuned_precision_not_merged": tuned_row["precision_not_merged"],
                "tuned_recall_not_merged": tuned_row["recall_not_merged"],
                "tuned_f1_not_merged": tuned_row["f1_not_merged"],
            }
        )
    return pd.DataFrame(rows)


def score_stress_split(
    name: str,
    df: pd.DataFrame,
    train_indices: np.ndarray,
    valid_indices: np.ndarray,
    model_name: str,
    feature_list: list[str],
) -> dict[str, object]:
    X_train = df.iloc[train_indices][feature_list]
    y_train = df.iloc[train_indices][TARGET_COLUMN]
    X_valid = df.iloc[valid_indices][feature_list]
    y_valid = df.iloc[valid_indices][TARGET_COLUMN]
    model = fit_pipeline(
        model_name,
        build_models(y_train, feature_list)[model_name],
        X_train,
        y_train,
    )
    row = score_pipeline(model_name, model, X_valid, y_valid)
    row["stress_test"] = name
    row["training_rows"] = len(X_train)
    row["validation_rows"] = len(X_valid)
    row["validation_not_merged_rate"] = (1 - y_valid.mean()) * 100
    return row


def generalization_stress_tests(
    train_df: pd.DataFrame,
    model_name: str,
    feature_list: list[str],
    sample_size: int = 220_000,
) -> pd.DataFrame:
    stress_df = stratified_sample(train_df[[TARGET_COLUMN, "project_id", "creator_id", "last_close_time", *feature_list]], sample_size, random_state=RANDOM_STATE + 99)
    stress_df = stress_df.reset_index(drop=True)
    rows = []

    random_train, random_valid = train_test_split(
        np.arange(len(stress_df)),
        test_size=0.25,
        stratify=stress_df[TARGET_COLUMN],
        random_state=RANDOM_STATE,
    )
    rows.append(score_stress_split("random_stratified_sample", stress_df, random_train, random_valid, model_name, feature_list))

    temporal_order = stress_df.sort_values("last_close_time").index.to_numpy()
    temporal_cut = int(len(temporal_order) * 0.75)
    rows.append(score_stress_split("temporal_last_25pct_sample", stress_df, temporal_order[:temporal_cut], temporal_order[temporal_cut:], model_name, feature_list))

    for group_column, stress_name in [
        ("project_id", "project_group_holdout_sample"),
        ("creator_id", "creator_group_holdout_sample"),
    ]:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
        train_idx, valid_idx = next(splitter.split(stress_df[feature_list], stress_df[TARGET_COLUMN], groups=stress_df[group_column]))
        rows.append(score_stress_split(stress_name, stress_df, train_idx, valid_idx, model_name, feature_list))

    return pd.DataFrame(rows)


def stress_model_family_comparison(
    train_df: pd.DataFrame,
    feature_list: list[str],
    sample_size: int = 160_000,
) -> pd.DataFrame:
    selected_features = list(feature_list)
    stress_df = stratified_sample(
        train_df[[TARGET_COLUMN, "project_id", "creator_id", "last_close_time", *selected_features]],
        sample_size,
        random_state=RANDOM_STATE + 505,
    ).reset_index(drop=True)

    split_rows: list[tuple[str, np.ndarray, np.ndarray]] = []
    random_train, random_valid = train_test_split(
        np.arange(len(stress_df)),
        test_size=0.25,
        stratify=stress_df[TARGET_COLUMN],
        random_state=RANDOM_STATE,
    )
    split_rows.append(("random_stratified_sample", random_train, random_valid))

    temporal_order = stress_df.sort_values("last_close_time").index.to_numpy()
    temporal_cut = int(len(temporal_order) * 0.75)
    split_rows.append(("temporal_last_25pct_sample", temporal_order[:temporal_cut], temporal_order[temporal_cut:]))

    for group_column, stress_name in [
        ("project_id", "project_group_holdout_sample"),
        ("creator_id", "creator_group_holdout_sample"),
    ]:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
        train_idx, valid_idx = next(
            splitter.split(stress_df[selected_features], stress_df[TARGET_COLUMN], groups=stress_df[group_column])
        )
        split_rows.append((stress_name, train_idx, valid_idx))

    rows = []
    for stress_name, train_idx, valid_idx in split_rows:
        X_train = stress_df.iloc[train_idx][selected_features]
        y_train = stress_df.iloc[train_idx][TARGET_COLUMN]
        X_valid = stress_df.iloc[valid_idx][selected_features]
        y_valid = stress_df.iloc[valid_idx][TARGET_COLUMN]
        for model_name, pipeline in build_models(y_train, selected_features).items():
            fitted = fit_pipeline(model_name, pipeline, X_train, y_train)
            row = score_pipeline(model_name, fitted, X_valid, y_valid)
            row["stress_test"] = stress_name
            row["training_rows"] = len(X_train)
            row["validation_rows"] = len(X_valid)
            row["validation_not_merged_rate"] = (1 - y_valid.mean()) * 100
            rows.append(row)
    result = pd.DataFrame(rows)
    result["rank_within_stress_f1"] = result.groupby("stress_test")["f1_not_merged"].rank(
        method="dense",
        ascending=False,
    )
    return result


def project_cluster_bootstrap_intervals(
    y_true: pd.Series,
    y_pred: np.ndarray,
    project_ids: pd.Series,
    threshold_label: str,
    point_estimates: dict[str, float],
    n_resamples: int = 300,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    frame = pd.DataFrame({"y": np.asarray(y_true), "pred": np.asarray(y_pred), "project_id": np.asarray(project_ids)})
    projects = frame["project_id"].drop_duplicates().to_numpy()
    group_indices = frame.groupby("project_id").indices
    samples = {"precision_not_merged": [], "recall_not_merged": [], "f1_not_merged": [], "balanced_accuracy": []}
    for _ in range(n_resamples):
        sampled_projects = rng.choice(projects, size=len(projects), replace=True)
        sampled_indices = np.concatenate([group_indices[project] for project in sampled_projects])
        sampled = frame.iloc[sampled_indices]
        y_boot = sampled["y"]
        pred_boot = sampled["pred"]
        samples["precision_not_merged"].append(precision_score(y_boot, pred_boot, pos_label=0, zero_division=0))
        samples["recall_not_merged"].append(recall_score(y_boot, pred_boot, pos_label=0, zero_division=0))
        samples["f1_not_merged"].append(f1_score(y_boot, pred_boot, pos_label=0, zero_division=0))
        samples["balanced_accuracy"].append(balanced_accuracy_score(y_boot, pred_boot))
    rows = []
    for metric, values in samples.items():
        clean = np.asarray(values)
        rows.append(
            {
                "threshold_label": threshold_label,
                "metric": metric,
                "estimate": point_estimates[metric],
                "cluster_bootstrap_mean": clean.mean(),
                "ci_lower_95": np.quantile(clean, 0.025),
                "ci_upper_95": np.quantile(clean, 0.975),
                "bootstrap_resamples": n_resamples,
                "cluster_unit": "project_id",
            }
        )
    return pd.DataFrame(rows)


def validation_permutation_importance(
    pipeline: Pipeline,
    X_valid: pd.DataFrame,
    y_valid: pd.Series,
    feature_list: list[str],
    sample_size: int = 35_000,
) -> pd.DataFrame:
    if len(X_valid) > sample_size:
        _, X_sample, _, y_sample = train_test_split(
            X_valid,
            y_valid,
            test_size=sample_size,
            stratify=y_valid,
            random_state=RANDOM_STATE,
        )
    else:
        X_sample, y_sample = X_valid, y_valid

    def f1_not_merged_scorer(estimator: Pipeline, X: pd.DataFrame, y: pd.Series) -> float:
        return f1_score(y, estimator.predict(X), pos_label=0, zero_division=0)

    result = permutation_importance(
        pipeline,
        X_sample,
        y_sample,
        scoring=f1_not_merged_scorer,
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    return (
        pd.DataFrame(
            {
                "feature": feature_list,
                "importance_mean": result.importances_mean,
                "importance_std": result.importances_std,
                "sample_size": len(X_sample),
                "scoring": "validation_not_merged_f1_drop",
            }
        )
        .sort_values("importance_mean", ascending=False)
        .reset_index(drop=True)
    )


def feature_family_ablation(
    train_df: pd.DataFrame,
    base_model_name: str,
    feature_list: list[str],
    validation_size: float = 0.25,
    sample_size: int = 260_000,
) -> pd.DataFrame:
    rows = []
    base_comparison, _, _ = fit_and_compare(
        train_df,
        sample_size,
        validation_size=validation_size,
        feature_list=feature_list,
        feature_policy="headline_leakage_safer_features",
    )
    base_row = base_comparison[base_comparison["model"] == base_model_name].iloc[0].to_dict()
    rows.append({"ablation": "none_full_headline", "removed_family": "none", "remaining_feature_count": len(feature_list), **base_row})
    for family, family_features in feature_family_groups.items():
        remaining_features = [feature for feature in feature_list if feature not in family_features]
        comparison, _, _ = fit_and_compare(
            train_df,
            sample_size,
            validation_size=validation_size,
            feature_list=remaining_features,
            feature_policy=f"ablation_without_{family}",
        )
        row = comparison[comparison["model"] == base_model_name].iloc[0].to_dict()
        rows.append(
            {
                "ablation": f"without_{family}",
                "removed_family": family,
                "remaining_feature_count": len(remaining_features),
                **row,
            }
        )
    result = pd.DataFrame(rows)
    baseline_f1 = result.loc[result["ablation"] == "none_full_headline", "f1_not_merged"].iloc[0]
    baseline_ap = result.loc[result["ablation"] == "none_full_headline", "average_precision_not_merged"].iloc[0]
    result["delta_f1_not_merged_vs_full"] = result["f1_not_merged"] - baseline_f1
    result["delta_average_precision_vs_full"] = result["average_precision_not_merged"] - baseline_ap
    return result


def split_id_integrity_table(train_path: Path, test_path: Path) -> pd.DataFrame:
    train_ids = pd.read_csv(train_path, usecols=["id"])["id"]
    test_ids = pd.read_csv(test_path, usecols=["id"])["id"]
    train_unique = train_ids.nunique()
    test_unique = test_ids.nunique()
    overlap = len(set(train_ids).intersection(set(test_ids)))
    return pd.DataFrame(
        [
            {"check": "train_rows", "value": len(train_ids), "interpretation": "Training PR rows loaded for id-integrity audit."},
            {"check": "train_unique_ids", "value": train_unique, "interpretation": "Unique PR ids in the training split."},
            {"check": "train_duplicate_ids", "value": int(train_ids.duplicated().sum()), "interpretation": "Duplicate PR ids inside the training split."},
            {"check": "test_rows", "value": len(test_ids), "interpretation": "Official test PR rows loaded for id-integrity audit."},
            {"check": "test_unique_ids", "value": test_unique, "interpretation": "Unique PR ids in the official test split."},
            {"check": "test_duplicate_ids", "value": int(test_ids.duplicated().sum()), "interpretation": "Duplicate PR ids inside the official test split."},
            {"check": "train_test_id_overlap", "value": overlap, "interpretation": "Exact PR ids shared by the train and test files."},
        ]
    )


def test_prediction_risk_bands(
    y_true: pd.Series,
    score_not_merged: np.ndarray,
    n_bins: int = 10,
) -> pd.DataFrame:
    frame = pd.DataFrame(
        {
            "actual_not_merged": 1 - np.asarray(y_true),
            "predicted_not_merged_score": np.asarray(score_not_merged),
        }
    )
    frame["risk_decile"] = pd.qcut(
        frame["predicted_not_merged_score"].rank(method="first"),
        q=n_bins,
        labels=False,
    ) + 1
    baseline_rate = frame["actual_not_merged"].mean()
    summary = (
        frame.groupby("risk_decile", as_index=False)
        .agg(
            row_count=("actual_not_merged", "size"),
            actual_not_merged=("actual_not_merged", "sum"),
            min_predicted_not_merged_score=("predicted_not_merged_score", "min"),
            mean_predicted_not_merged_score=("predicted_not_merged_score", "mean"),
            max_predicted_not_merged_score=("predicted_not_merged_score", "max"),
            actual_not_merged_rate=("actual_not_merged", "mean"),
        )
        .sort_values("risk_decile")
        .reset_index(drop=True)
    )
    summary["risk_label"] = np.where(
        summary["risk_decile"] == n_bins,
        "highest predicted not-merged risk",
        np.where(summary["risk_decile"] == 1, "lowest predicted not-merged risk", "middle risk band"),
    )
    summary["baseline_not_merged_rate"] = baseline_rate
    summary["lift_vs_baseline"] = summary["actual_not_merged_rate"] / baseline_rate
    summary["actual_not_merged_rate_pct"] = summary["actual_not_merged_rate"] * 100
    summary["baseline_not_merged_rate_pct"] = baseline_rate * 100
    return summary[
        [
            "risk_decile",
            "risk_label",
            "row_count",
            "actual_not_merged",
            "min_predicted_not_merged_score",
            "mean_predicted_not_merged_score",
            "max_predicted_not_merged_score",
            "actual_not_merged_rate",
            "actual_not_merged_rate_pct",
            "baseline_not_merged_rate",
            "baseline_not_merged_rate_pct",
            "lift_vs_baseline",
        ]
    ]


def calibration_summary_table(
    y_true: pd.Series,
    score_not_merged: np.ndarray,
    n_bins: int = 10,
) -> pd.DataFrame:
    observed = 1 - np.asarray(y_true)
    scores = np.asarray(score_not_merged)
    frame = pd.DataFrame(
        {
            "actual_not_merged": observed,
            "predicted_not_merged_score": scores,
        }
    )
    frame["calibration_bin"] = pd.qcut(
        frame["predicted_not_merged_score"].rank(method="first"),
        q=n_bins,
        labels=False,
    ) + 1
    brier_score = float(np.mean((scores - observed) ** 2))
    summary = (
        frame.groupby("calibration_bin", as_index=False)
        .agg(
            row_count=("actual_not_merged", "size"),
            min_predicted_not_merged_score=("predicted_not_merged_score", "min"),
            mean_predicted_not_merged_score=("predicted_not_merged_score", "mean"),
            max_predicted_not_merged_score=("predicted_not_merged_score", "max"),
            observed_not_merged_rate=("actual_not_merged", "mean"),
        )
        .sort_values("calibration_bin")
        .reset_index(drop=True)
    )
    summary["calibration_error"] = summary["observed_not_merged_rate"] - summary["mean_predicted_not_merged_score"]
    summary["absolute_calibration_error"] = summary["calibration_error"].abs()
    summary["brier_score_not_merged"] = brier_score
    summary["observed_not_merged_rate_pct"] = summary["observed_not_merged_rate"] * 100
    summary["mean_predicted_not_merged_score_pct"] = summary["mean_predicted_not_merged_score"] * 100
    return summary


def weighted_abs_calibration_error(y_true: pd.Series, score_not_merged: np.ndarray, n_bins: int = 10) -> float:
    calibration = calibration_summary_table(y_true, score_not_merged, n_bins=n_bins)
    return float(
        np.average(
            calibration["absolute_calibration_error"],
            weights=calibration["row_count"],
        )
    )


def evaluate_prediction_contracts(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    model_name: str,
) -> pd.DataFrame:
    rows = []
    for contract_name, contract in prediction_contracts.items():
        features = list(contract["features"])
        X = train_df[features]
        y = train_df[TARGET_COLUMN]
        X_train, X_valid, y_train, y_valid = train_test_split(
            X,
            y,
            test_size=0.25,
            stratify=y,
            random_state=RANDOM_STATE,
        )
        validation_model = fit_pipeline(
            model_name,
            build_models(y_train, features)[model_name],
            X_train,
            y_train,
        )
        validation_row = add_metric_metadata(
            score_pipeline(model_name, validation_model, X_valid, y_valid),
            feature_policy=contract_name,
            feature_count=len(features),
            source_rows=len(train_df),
            training_rows=len(X_train),
            validation_rows=len(X_valid),
            training_scope="full_source_internal_validation",
        )
        validation_row["contract"] = contract_name
        validation_row["contract_label"] = contract["label"]
        validation_row["evaluation_split"] = "validation"
        validation_row["interpretation"] = contract["interpretation"]
        rows.append(validation_row)

        final_model = fit_pipeline(
            model_name,
            build_models(y, features)[model_name],
            X,
            y,
        )
        test_row = add_metric_metadata(
            score_pipeline(model_name, final_model, test_df[features], test_df[TARGET_COLUMN]),
            feature_policy=contract_name,
            feature_count=len(features),
            source_rows=len(train_df),
            training_rows=len(train_df),
            validation_rows=len(test_df),
            training_scope="full_training_split",
        )
        test_row["contract"] = contract_name
        test_row["contract_label"] = contract["label"]
        test_row["evaluation_split"] = "test"
        test_row["interpretation"] = contract["interpretation"]
        rows.append(test_row)
    return pd.DataFrame(rows)


def calibration_model_comparison(
    train_df: pd.DataFrame,
    model_name: str,
    feature_list: list[str],
    sample_size: int = 180_000,
) -> pd.DataFrame:
    calibration_df = stratified_sample(
        train_df[[TARGET_COLUMN, *feature_list]],
        sample_size,
        random_state=RANDOM_STATE + 707,
    )
    train_part, holdout_part = train_test_split(
        calibration_df,
        test_size=0.40,
        stratify=calibration_df[TARGET_COLUMN],
        random_state=RANDOM_STATE,
    )
    calibrate_part, valid_part = train_test_split(
        holdout_part,
        test_size=0.50,
        stratify=holdout_part[TARGET_COLUMN],
        random_state=RANDOM_STATE,
    )
    X_train = train_part[feature_list]
    y_train = train_part[TARGET_COLUMN]
    X_calibrate = calibrate_part[feature_list]
    y_calibrate = calibrate_part[TARGET_COLUMN]
    X_valid = valid_part[feature_list]
    y_valid = valid_part[TARGET_COLUMN]

    base_model = fit_pipeline(
        model_name,
        build_models(y_train, feature_list)[model_name],
        X_train,
        y_train,
    )
    estimators: list[tuple[str, object]] = [("uncalibrated", base_model)]
    for method in ["sigmoid", "isotonic"]:
        calibrated = CalibratedClassifierCV(
            estimator=FrozenEstimator(base_model),
            method=method,
        )
        calibrated.fit(X_calibrate, y_calibrate)
        estimators.append((method, calibrated))

    rows = []
    for method, estimator in estimators:
        score_merged = estimator.predict_proba(X_valid)[:, 1]
        score_not_merged = 1 - score_merged
        predictions = estimator.predict(X_valid)
        row = score_predictions(
            model_name,
            y_valid,
            predictions,
            score_merged,
            threshold_label=f"{method}_default",
        )
        row["calibration_method"] = method
        row["model_sample_size"] = len(calibration_df)
        row["training_rows"] = len(train_part)
        row["calibration_rows"] = len(calibrate_part)
        row["validation_rows"] = len(valid_part)
        row["brier_score_not_merged"] = brier_score_loss(1 - y_valid, score_not_merged)
        row["weighted_abs_calibration_error"] = weighted_abs_calibration_error(y_valid, score_not_merged)
        rows.append(row)
    return pd.DataFrame(rows)


def full_generalization_benchmarks(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    model_name: str,
    feature_list: list[str],
    sample_size: int = 500_000,
) -> pd.DataFrame:
    rows = []
    final_model = fit_pipeline(
        model_name,
        build_models(train_df[TARGET_COLUMN], feature_list)[model_name],
        train_df[feature_list],
        train_df[TARGET_COLUMN],
    )
    official = score_pipeline(model_name, final_model, test_df[feature_list], test_df[TARGET_COLUMN])
    official["benchmark"] = "official_test"
    official["benchmark_scope"] = "official_train_test_files"
    official["training_rows"] = len(train_df)
    official["validation_rows"] = len(test_df)
    official["validation_not_merged_rate"] = (1 - test_df[TARGET_COLUMN].mean()) * 100
    rows.append(official)

    benchmark_df = stratified_sample(
        train_df[[TARGET_COLUMN, "project_id", "creator_id", "last_close_time", *feature_list]],
        sample_size,
        random_state=RANDOM_STATE + 808,
    ).reset_index(drop=True)
    temporal_order = benchmark_df.sort_values("last_close_time").index.to_numpy()
    temporal_cut = int(len(temporal_order) * 0.75)
    split_specs: list[tuple[str, np.ndarray, np.ndarray]] = [
        ("temporal_holdout", temporal_order[:temporal_cut], temporal_order[temporal_cut:]),
    ]
    for group_column, benchmark in [
        ("project_id", "project_holdout"),
        ("creator_id", "creator_holdout"),
    ]:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
        train_idx, valid_idx = next(
            splitter.split(
                benchmark_df[feature_list],
                benchmark_df[TARGET_COLUMN],
                groups=benchmark_df[group_column],
            )
        )
        split_specs.append((benchmark, train_idx, valid_idx))

    for benchmark, train_idx, valid_idx in split_specs:
        row = score_stress_split(benchmark, benchmark_df, train_idx, valid_idx, model_name, feature_list)
        row["benchmark"] = benchmark
        row["benchmark_scope"] = f"{len(benchmark_df)}_row_large_holdout"
        rows.append(row)
    return pd.DataFrame(rows)


def paired_model_delta_intervals(
    fitted_models: dict[str, Pipeline],
    X_valid: pd.DataFrame,
    y_valid: pd.Series,
    baseline_model: str,
    comparison_model: str,
    n_resamples: int = 400,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    y_array = np.asarray(y_valid)
    base_pred = fitted_models[baseline_model].predict(X_valid)
    comp_pred = fitted_models[comparison_model].predict(X_valid)
    base_score = score_merged_probability(fitted_models[baseline_model], X_valid)
    comp_score = score_merged_probability(fitted_models[comparison_model], X_valid)

    def metric_values(indices: np.ndarray) -> dict[str, float]:
        y_boot = y_array[indices]
        return {
            "f1_not_merged": (
                f1_score(y_boot, base_pred[indices], pos_label=0, zero_division=0)
                - f1_score(y_boot, comp_pred[indices], pos_label=0, zero_division=0)
            ),
            "average_precision_not_merged": (
                average_precision_score(1 - y_boot, 1 - base_score[indices])
                - average_precision_score(1 - y_boot, 1 - comp_score[indices])
            ),
        }

    full = metric_values(np.arange(len(y_array)))
    samples = {metric: [] for metric in full}
    for _ in range(n_resamples):
        indices = rng.integers(0, len(y_array), len(y_array))
        boot = metric_values(indices)
        for metric, value in boot.items():
            samples[metric].append(value)
    rows = []
    for metric, delta in full.items():
        values = np.asarray(samples[metric])
        rows.append(
            {
                "baseline_model": baseline_model,
                "comparison_model": comparison_model,
                "metric": metric,
                "delta": delta,
                "ci_lower_95": float(np.quantile(values, 0.025)),
                "ci_upper_95": float(np.quantile(values, 0.975)),
                "bootstrap_mean": float(values.mean()),
                "bootstrap_resamples": n_resamples,
                "interpretation": "Positive delta favors the selected Random Forest baseline over the comparison model.",
            }
        )
    return pd.DataFrame(rows)


def error_profile_summary_table(
    X: pd.DataFrame,
    y_true: pd.Series,
    y_pred: np.ndarray,
    score_not_merged: np.ndarray,
) -> pd.DataFrame:
    frame = X.copy()
    frame["actual"] = np.asarray(y_true)
    frame["predicted"] = np.asarray(y_pred)
    frame["actual_not_merged"] = 1 - frame["actual"]
    frame["predicted_not_merged"] = 1 - frame["predicted"]
    frame["predicted_not_merged_score"] = np.asarray(score_not_merged)
    frame["outcome_group"] = np.select(
        [
            (frame["actual"] == 0) & (frame["predicted"] == 0),
            (frame["actual"] == 0) & (frame["predicted"] == 1),
            (frame["actual"] == 1) & (frame["predicted"] == 0),
            (frame["actual"] == 1) & (frame["predicted"] == 1),
        ],
        [
            "correct_not_merged",
            "missed_not_merged",
            "false_not_merged",
            "correct_merged",
        ],
        default="unknown",
    )
    total_rows = len(frame)
    summary = (
        frame.groupby("outcome_group", as_index=False)
        .agg(
            row_count=("actual", "size"),
            actual_not_merged_rate=("actual_not_merged", "mean"),
            predicted_not_merged_rate=("predicted_not_merged", "mean"),
            mean_predicted_not_merged_score=("predicted_not_merged_score", "mean"),
            first_pr_rate=("first_pr", "mean"),
            non_core_rate=("core_member", lambda values: 1 - values.mean()),
            ci_exists_rate=("ci_exists", "mean"),
            test_inclusion_rate=("test_inclusion", "mean"),
            median_prev_pullreqs=("prev_pullreqs", "median"),
            median_contrib_perc_commit=("contrib_perc_commit", "median"),
            median_open_pr_num=("open_pr_num", "median"),
            median_stars=("stars", "median"),
            median_churn_addition=("churn_addition", "median"),
            median_description_length=("description_length", "median"),
        )
        .sort_values("outcome_group")
        .reset_index(drop=True)
    )
    summary["share_of_test_pct"] = summary["row_count"] / total_rows * 100
    order = {
        "correct_not_merged": 1,
        "missed_not_merged": 2,
        "false_not_merged": 3,
        "correct_merged": 4,
    }
    summary["display_order"] = summary["outcome_group"].map(order)
    return summary.sort_values("display_order").reset_index(drop=True)


def error_analysis_findings_table(
    risk_bands: pd.DataFrame,
    calibration: pd.DataFrame,
    error_profile: pd.DataFrame,
) -> pd.DataFrame:
    top_risk = risk_bands.loc[risk_bands["risk_decile"].idxmax()]
    low_risk = risk_bands.loc[risk_bands["risk_decile"].idxmin()]
    missed = error_profile[error_profile["outcome_group"] == "missed_not_merged"].iloc[0]
    caught = error_profile[error_profile["outcome_group"] == "correct_not_merged"].iloc[0]
    false_alarm = error_profile[error_profile["outcome_group"] == "false_not_merged"].iloc[0]
    correct_merged = error_profile[error_profile["outcome_group"] == "correct_merged"].iloc[0]
    mean_abs_calibration_error = np.average(
        calibration["absolute_calibration_error"],
        weights=calibration["row_count"],
    )
    brier_score = calibration["brier_score_not_merged"].iloc[0]
    return pd.DataFrame(
        [
            {
                "finding": "Predicted risk ranks concentrate not-merged outcomes",
                "evidence": (
                    f"Top risk decile actual not-merged rate {top_risk['actual_not_merged_rate_pct']:.2f}% "
                    f"vs baseline {top_risk['baseline_not_merged_rate_pct']:.2f}% "
                    f"and lowest decile {low_risk['actual_not_merged_rate_pct']:.2f}%."
                ),
                "interpretation": "The model is more defensible as a risk-ranking signal than as a precise decision rule.",
            },
            {
                "finding": "Missed not-merged PRs receive lower risk scores than caught not-merged PRs",
                "evidence": (
                    f"Mean not-merged score {missed['mean_predicted_not_merged_score']:.3f} for missed not-merged PRs "
                    f"vs {caught['mean_predicted_not_merged_score']:.3f} for correctly flagged not-merged PRs."
                ),
                "interpretation": "The main failure mode is ambiguous not-merged cases that look similar to merged PRs under available features.",
            },
            {
                "finding": "False not-merged predictions are concentrated in busier or larger project contexts",
                "evidence": (
                    f"False not-merged median open PRs {false_alarm['median_open_pr_num']:.0f} and stars {false_alarm['median_stars']:.0f} "
                    f"vs correct-merged medians {correct_merged['median_open_pr_num']:.0f} and {correct_merged['median_stars']:.0f}."
                ),
                "interpretation": "Project context is useful but can make accepted PRs in large/busy repositories look risky.",
            },
            {
                "finding": "Scores are imperfectly calibrated",
                "evidence": (
                    f"Weighted mean absolute calibration error {mean_abs_calibration_error:.3f}; "
                    f"Brier score for not-merged score {brier_score:.3f}."
                ),
                "interpretation": "Predicted scores should be treated as ranking evidence, not literal probabilities.",
            },
        ]
    )


In [ ]:
print(f"Project root: {PROJECT_ROOT}")
print(f"Train path exists: {TRAIN_PATH.exists()}")
print(f"Test path exists: {TEST_PATH.exists()}")


## Leakage-aware feature set

The model uses the headline leakage-safer feature policy carried into the final notebook. `language` is treated as a categorical code, not a rank. Review/process variables, close-time PR evolution fields, and target-adjacent success-rate fields whose timing is unclear stay out of this checkpoint model.


In [ ]:
feature_groups = pd.DataFrame(
    [
        {"group": "Identifiers", "count": len(exclude_ids), "use": "Excluded"},
        {"group": "Post-outcome leakage", "count": len(exclude_post_outcome), "use": "Excluded"},
        {"group": "Timing-sensitive under review", "count": len(ambiguous_features), "use": "Held back"},
        {"group": "Headline leakage-safer modeling features", "count": len(headline_safe_features), "use": "Used now"},
        {"group": "Integrator-assumed extension features", "count": len(integrator_assumed_extension_features), "use": "Held for sensitivity"},
        {"group": "Timing-assumed extension features", "count": len(timing_assumed_extension_features), "use": "Held for sensitivity"},
    ]
)
safe_feature_table = pd.DataFrame(
    {
        "feature": headline_safe_features,
        "availability_rationale": [availability_reason(feature) for feature in headline_safe_features],
    }
)
display(feature_groups)
display(safe_feature_table)


## Load the training split and audit the target


In [ ]:
train_df = pd.read_csv(TRAIN_PATH, usecols=safe_usecols, low_memory=False)
test_header = pd.read_csv(TEST_PATH, nrows=0)

audit_summary = pd.DataFrame(
    [
        {"check": "training rows loaded", "result": f"{len(train_df):,}"},
        {"check": "headline leakage-safer feature columns", "result": len(headline_safe_features)},
        {"check": "target present", "result": TARGET_COLUMN in train_df.columns},
        {"check": "explicit nulls in modeling frame", "result": int(train_df.isna().sum().sum())},
        {"check": "duplicate rows in modeling frame", "result": int(train_df.duplicated().sum())},
        {"check": "test schema contains target", "result": TARGET_COLUMN in test_header.columns},
    ]
)
display(audit_summary)
display(target_distribution(train_df, "train"))


## Baseline model comparison


In [ ]:
MODEL_SAMPLE_SIZE = 220_000
comparison, fitted_models, validation_data = fit_and_compare(train_df, MODEL_SAMPLE_SIZE)
display(comparison)

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = comparison.melt(
    id_vars="model",
    value_vars=["balanced_accuracy", "f1_not_merged", "average_precision_not_merged"],
    var_name="metric",
    value_name="score",
)
sns.barplot(data=plot_df, x="score", y="model", hue="metric", ax=ax)
ax.set_title("Checkpoint 2 validation comparison on a stratified training sample")
ax.set_xlim(0, 1)
ax.set_xlabel("Score")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## Interpretation for Checkpoint 2

The dummy classifier is expected to look deceptively strong on raw accuracy because merged PRs dominate the dataset. For grading and scientific credibility, the minority-class metrics are more informative: recall and F1 for not-merged PRs, balanced accuracy, and average precision for the not-merged class.


In [ ]:
best_row = comparison.iloc[0]
display(Markdown(f'''
**Checkpoint 2 finding.** The strongest validation baseline by not-merged F1 is **{best_row["model"]}**.

- Not-merged F1: `{best_row["f1_not_merged"]:.3f}`
- Not-merged recall: `{best_row["recall_not_merged"]:.3f}`
- Balanced accuracy: `{best_row["balanced_accuracy"]:.3f}`
- Not-merged average precision: `{best_row["average_precision_not_merged"]:.3f}`

The next checkpoint should lock the final feature set, run the selected model against the untouched test split, and add one unsupervised analysis that profiles PR groups without using the target during clustering.
'''))


## Checkpoint 2 limitations

- This is a validation baseline, not the final model.
- The provided test split is intentionally not used here.
- Timing-sensitive discussion, CI, and sentiment fields are still held back until their availability at prediction time can be defended.
- The baseline uses a stratified sample for runtime; the final notebook should report the sample size and evaluate on the full provided test split.
